# Mini Project 1 — Section 4: Visualization

Run the **Setup** cell first, then the visualization cells below.


In [ ]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

**First visualization — state comparison:**  
Hospital capacity pressure over time (inpatient vs ICU occupancy) for states with the highest recent inpatient occupancy.

In [19]:
# Section 4 visualization using Plotly Express
import pandas as pd
import plotly.express as px
from pathlib import Path

csv_candidates = [
    Path("MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../MP1 Project/MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../530 Week5 Project/Weekly_Hospital_Respiratory_Data.csv"),
    Path("Weekly_Hospital_Respiratory_Data.csv"),
]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find hospital respiratory CSV. Tried: "
        + ", ".join(str(p) for p in csv_candidates)
    )

resp = pd.read_csv(csv_path, low_memory=False)

state_codes = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "DC"
}

capacity_cols = [
    "Percent Inpatient Beds Occupied",
    "Percent ICU Beds Occupied",
]

resp["Week Ending Date"] = pd.to_datetime(resp["Week Ending Date"], errors="coerce")
for col in capacity_cols:
    resp[col] = (
        resp[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    resp[col] = pd.to_numeric(resp[col], errors="coerce")

state_df = resp[
    resp["Geographic aggregation"].isin(state_codes)
].dropna(subset=["Week Ending Date"] + capacity_cols).copy()
state_df = state_df.sort_values("Week Ending Date")

# Focus on top states by latest inpatient occupancy for readable comparison
latest_week = state_df["Week Ending Date"].max()
latest_slice = state_df[state_df["Week Ending Date"] == latest_week]
top_states = (
    latest_slice
    .nlargest(12, "Percent Inpatient Beds Occupied")["Geographic aggregation"]
    .tolist()
)
plot_df = state_df[state_df["Geographic aggregation"].isin(top_states)].copy()

long_df = plot_df.melt(
    id_vars=["Week Ending Date", "Geographic aggregation"],
    value_vars=capacity_cols,
    var_name="Metric",
    value_name="OccupancyPercent",
)

metric_labels = {
    "Percent Inpatient Beds Occupied": "Inpatient beds occupied (%)",
    "Percent ICU Beds Occupied": "ICU beds occupied (%)",
}
long_df["Metric"] = long_df["Metric"].map(metric_labels)

fig = px.line(
    long_df,
    x="Week Ending Date",
    y="OccupancyPercent",
    color="Geographic aggregation",
    facet_row="Metric",
    title=None,
    labels={
        "Week Ending Date": "Reporting week",
        "OccupancyPercent": "Occupancy (%)",
        "Geographic aggregation": "State",
        "Metric": "Capacity measure",
    },
    height=900,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(
    title=dict(
        text=(
            "<b>Hospital capacity pressure varies across high-burden states</b><br>"
            "<span style='font-size:12px;color:#444'>"
            "Top 12 states by most recent inpatient occupancy. "
            "Top row: inpatient beds. Bottom row: ICU beds."
            "</span>"
        ),
        x=0.5,
        xanchor="center",
        pad=dict(b=18),
    ),
    margin=dict(t=170, r=40, b=70, l=70),
    legend_title_text="State",
    title_automargin=True,
)
fig.update_traces(
    hovertemplate=(
        "State: %{fullData.name}<br>"
        "Reporting week: %{x|%b %d, %Y}<br>"
        "Occupancy: %{y:.1f}%<extra></extra>"
    )
)
fig.show()

png_path = Path("viz1_state_hospital_capacity.png")
fig.write_image(png_path, engine="kaleido", scale=2)
print(f"Saved {png_path.resolve()}")

**Chart rationale (first visualization — state comparison):**  
I used a **line chart** because this is weekly time-series data, and lines are the clearest way to compare how hospital capacity pressure changes over time across states. I faceted the chart into inpatient and ICU occupancy so the reader can compare both capacity measures without mixing scales.

I also focused on the states with the highest latest-week inpatient occupancy to keep the chart readable while still highlighting where sustained pressure is strongest. The main takeaway is that capacity pressure is not uniform: several states remain persistently high over time, especially when comparing inpatient and ICU occupancy together.

**Second visualization — research question:**  
Are there seasonal or temporal patterns in hospital occupancy and bed utilization? *(Analyzing weekly trends to detect recurring peaks or cycles in hospital demand.)*

In [20]:
# Second visualization: seasonal / temporal patterns (national USA)
# Heatmap (year × month): same calendar month lines up across years for seasonality.

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# COLOR SCALE: 60% = bright yellow, 80% = deep red
YELLOW_TO_RED = [
    [0.0, "#FFEB3B"],
    [0.25, "#FFC107"],
    [0.5, "#FF9800"],
    [0.75, "#F44336"],
    [1.0, "#B71C1C"],
]
ZMIN_PCT, ZMAX_PCT = 60.0, 80.0

csv_candidates = [
    Path("MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../MP1 Project/MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../530 Week5 Project/Weekly_Hospital_Respiratory_Data.csv"),
    Path("Weekly_Hospital_Respiratory_Data.csv"),
]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find hospital respiratory CSV. Tried: "
        + ", ".join(str(p) for p in csv_candidates)
    )

resp_seasonal = pd.read_csv(csv_path, low_memory=False)
capacity_cols = [
    "Percent Inpatient Beds Occupied",
    "Percent ICU Beds Occupied",
]
resp_seasonal["Week Ending Date"] = pd.to_datetime(
    resp_seasonal["Week Ending Date"], errors="coerce"
)
for col in capacity_cols:
    resp_seasonal[col] = (
        resp_seasonal[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    resp_seasonal[col] = pd.to_numeric(resp_seasonal[col], errors="coerce")

usa = resp_seasonal[resp_seasonal["Geographic aggregation"] == "USA"].copy()
usa = usa.dropna(
    subset=["Week Ending Date", "Percent Inpatient Beds Occupied", "Percent ICU Beds Occupied"]
).sort_values("Week Ending Date")

usa["year"] = usa["Week Ending Date"].dt.year
usa["month"] = usa["Week Ending Date"].dt.month
monthly = (
    usa.groupby(["year", "month"], as_index=False)[capacity_cols]
    .mean()
    .rename(
        columns={
            "Percent Inpatient Beds Occupied": "inpatient_pct",
            "Percent ICU Beds Occupied": "icu_pct",
        }
    )
)

p_in = monthly.pivot(index="year", columns="month", values="inpatient_pct")
p_icu = monthly.pivot(index="year", columns="month", values="icu_pct")

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
x_labels = [month_names[m - 1] for m in p_in.columns]

fig_seasonal = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("Inpatient (% occupied)", "ICU (% occupied)"),
    vertical_spacing=0.16,
)

tick_vals = [60, 65, 70, 75, 80]
tick_text = ["60", "65", "70", "75", "80"]

fig_seasonal.add_trace(
    go.Heatmap(
        z=p_in.values,
        x=x_labels,
        y=p_in.index.astype(str),
        zmin=ZMIN_PCT,
        zmax=ZMAX_PCT,
        zsmooth=False,
        xgap=2,
        ygap=2,
        colorscale=YELLOW_TO_RED,
        hovertemplate="Year: %{y}<br>Month: %{x}<br>Inpatient: %{z:.1f}%<extra></extra>",
        colorbar=dict(
            title="Occupancy (%)",
            tickvals=tick_vals,
            ticktext=tick_text,
            len=0.42,
            y=0.78,
            yanchor="middle",
        ),
    ),
    row=1,
    col=1,
)
fig_seasonal.add_trace(
    go.Heatmap(
        z=p_icu.values,
        x=x_labels,
        y=p_icu.index.astype(str),
        zmin=ZMIN_PCT,
        zmax=ZMAX_PCT,
        zsmooth=False,
        xgap=2,
        ygap=2,
        colorscale=YELLOW_TO_RED,
        hovertemplate="Year: %{y}<br>Month: %{x}<br>ICU: %{z:.1f}%<extra></extra>",
        colorbar=dict(
            title="Occupancy (%)",
            tickvals=tick_vals,
            ticktext=tick_text,
            len=0.42,
            y=0.22,
            yanchor="middle",
        ),
    ),
    row=2,
    col=1,
)

fig_seasonal.update_xaxes(showgrid=False, zeroline=False, side="bottom", type="category", row=1, col=1)
fig_seasonal.update_xaxes(showgrid=False, zeroline=False, side="bottom", type="category", row=2, col=1)
fig_seasonal.update_yaxes(showgrid=False, zeroline=False, type="category", row=1, col=1)
fig_seasonal.update_yaxes(showgrid=False, zeroline=False, type="category", row=2, col=1)

fig_seasonal.update_layout(
    title=dict(
        text=(
            "<b>USA: seasonal occupancy (month × year)</b>"
            "<br><sup>60% = yellow, 80% = red</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=15),
        pad=dict(b=12),
    ),
    height=780,
    margin=dict(l=88, r=120, b=70, t=165),
)

print(f"Scale: {ZMIN_PCT}% (yellow) → {ZMAX_PCT}% (red); colorscale has {len(YELLOW_TO_RED)} stops")
fig_seasonal.show()

png_path = Path("viz2_seasonal_heatmap.png")
fig_seasonal.write_image(png_path, engine="kaleido", scale=2)
print(f"Saved {png_path.resolve()}")

Scale: 60.0% (yellow) → 80.0% (red); colorscale has 5 stops


**Chart rationale (second visualization):**  
**Why not only a line chart?** A single national line over time shows *when* demand spikes, but it is harder to see whether spikes **line up on the same calendar months across different years** (the core of “seasonal” structure). Multiple smoothed lines help, but they still ask the reader to mentally compare distant dates on one long axis.

**What I used instead:** A **year × month heatmap** (two panels: inpatient and ICU). Each row is a calendar year and each column is a month, with color showing the **average of weekly values** in that month for the USA. Recurring seasonal pressure shows up as **vertical bands** (for example darker colors in late fall and winter) that repeat across rows. That matches the question about **recurring peaks or cycles** more directly than a line chart alone.

**Reading the colors:** Both panels share the **same fixed scale: 60–80%** occupancy. **Yellow** marks the low end (60%) and **red** the high end (80%), with orange in between, so month-to-month differences show up clearly. Values below 60% clip to yellow; above 80% clip to red.

**Layout note:** Background grid lines are turned off because Plotly’s default Cartesian grid does not line up with heatmap cells and can look like a misaligned checkerboard.

**Title:** The title is a normal Plotly **layout title** with a larger **top margin (`t=165`)** so it stays inside the figure canvas in the notebook viewer.

**Takeaway:** Look for columns that stay darker across several years — that is evidence of a seasonal rhythm in national hospital bed use, separate from one-off events.

**Third visualization — research question:**  
Do pediatric and adult bed occupancy rates follow different patterns over time, and are pediatric ICU beds proportionally more strained than adult ICU beds?


In [18]:
# Third visualization: monthly trends — adult vs pediatric occupancy (USA)
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

csv_candidates = [
    Path("MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../MP1 Project/MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../530 Week5 Project/Weekly_Hospital_Respiratory_Data.csv"),
    Path("Weekly_Hospital_Respiratory_Data.csv"),
]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find hospital respiratory CSV")

usa = pd.read_csv(csv_path, low_memory=False)
usa = usa[usa["Geographic aggregation"] == "USA"].copy()

bed_cols = [
    "Number of Adult Inpatient Beds",
    "Number of Pediatric Inpatient beds",
    "Number of Adult Inpatient Beds Occupied",
    "Number of Pediatric Inpatient Beds Occupied",
    "Number of Adult ICU Beds",
    "Number of Pediatric ICU Beds",
    "Number of Adult ICU Beds Occupied",
    "Number of Pediatric ICU Beds Occupied",
]
usa["Week Ending Date"] = pd.to_datetime(usa["Week Ending Date"], errors="coerce")
for col in bed_cols:
    usa[col] = pd.to_numeric(
        usa[col].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )
usa = usa.dropna(subset=["Week Ending Date"]).sort_values("Week Ending Date")

usa["adult_inpatient_pct"] = usa["Number of Adult Inpatient Beds Occupied"] / usa["Number of Adult Inpatient Beds"] * 100
usa["ped_inpatient_pct"] = usa["Number of Pediatric Inpatient Beds Occupied"] / usa["Number of Pediatric Inpatient beds"] * 100
usa["adult_icu_pct"] = usa["Number of Adult ICU Beds Occupied"] / usa["Number of Adult ICU Beds"] * 100
usa["ped_icu_pct"] = usa["Number of Pediatric ICU Beds Occupied"] / usa["Number of Pediatric ICU Beds"] * 100

usa["month"] = usa["Week Ending Date"].dt.to_period("M").dt.to_timestamp()
monthly = usa.groupby("month", as_index=False)[
    ["adult_inpatient_pct", "ped_inpatient_pct", "adult_icu_pct", "ped_icu_pct"]
].mean()
monthly["month_label"] = monthly["month"].dt.strftime("%b %Y")

fig_monthly = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.22,
    subplot_titles=(
        "Inpatient: adult vs pediatric occupancy by month",
        "ICU: adult vs pediatric occupancy by month",
    ),
)

for adult_col, ped_col, adult_name, ped_name, row in [
    ("adult_inpatient_pct", "ped_inpatient_pct", "Adult inpatient", "Pediatric inpatient", 1),
    ("adult_icu_pct", "ped_icu_pct", "Adult ICU", "Pediatric ICU", 2),
]:
    fig_monthly.add_trace(
        go.Scatter(
            x=monthly["month_label"],
            y=monthly[adult_col],
            name=adult_name,
            mode="lines+markers",
            line=dict(color="#1976D2", width=2),
            marker=dict(size=5),
        ),
        row=row,
        col=1,
    )
    fig_monthly.add_trace(
        go.Scatter(
            x=monthly["month_label"],
            y=monthly[ped_col],
            name=ped_name,
            mode="lines+markers",
            line=dict(color="#FF7043", width=2),
            marker=dict(size=5),
        ),
        row=row,
        col=1,
    )

fig_monthly.update_yaxes(title_text="Occupancy (%)", row=1, col=1, range=[50, 90])
fig_monthly.update_yaxes(title_text="Occupancy (%)", row=2, col=1, range=[50, 90])
# Show month labels on BOTH panels (shared_xaxes hides top ticks by default)
for row in (1, 2):
    fig_monthly.update_xaxes(
        title_text="",
        showticklabels=True,
        tickangle=-45,
        type="category",
        row=row,
        col=1,
    )

fig_monthly.update_layout(
    title=dict(
        text="Adult vs pediatric hospital occupancy by month (USA)",
        x=0.5,
        xanchor="center",
        y=0.99,
    ),
    height=700,
    width=960,
    hovermode="x unified",
    template="plotly_white",
    legend=dict(
        orientation="h",
        y=-0.12,
        x=0.5,
        xanchor="center",
        yanchor="top",
    ),
    margin=dict(t=90, b=160, l=60, r=40),
)
fig_monthly.show()

png_path = Path("viz3_pediatric_adult_monthly.png")
fig_monthly.write_image(png_path, engine="kaleido", scale=2)
print(f"Saved {png_path.resolve()}")

avg_adult_inp = usa["adult_inpatient_pct"].mean()
avg_ped_inp = usa["ped_inpatient_pct"].mean()
avg_adult_icu = usa["adult_icu_pct"].mean()
avg_ped_icu = usa["ped_icu_pct"].mean()
icu_ratio = avg_ped_icu / avg_adult_icu

print("=" * 60)
print("AVERAGE OCCUPANCY (all weeks, USA — use for your write-up)")
print("=" * 60)
print(f"  Adult inpatient:       {avg_adult_inp:.1f}%")
print(f"  Pediatric inpatient: {avg_ped_inp:.1f}%")
print(f"  Adult ICU:             {avg_adult_icu:.1f}%")
print(f"  Pediatric ICU:         {avg_ped_icu:.1f}%")
print()
print("Q1: Do patterns differ over time?")
print("  → Compare blue vs orange in EACH panel month by month.")
print("  → Lines that peak in different months or move apart = different patterns.")
print()
print("Q2: Is pediatric ICU proportionally more strained than adult ICU?")
if icu_ratio > 1:
    print(f"  → YES (pediatric ICU average is higher; ratio {icu_ratio:.2f})")
else:
    print(f"  → NO (pediatric ICU average is lower; ratio {icu_ratio:.2f})")
print("=" * 60)



AVERAGE OCCUPANCY (all weeks, USA — use for your write-up)
  Adult inpatient:       75.5%
  Pediatric inpatient: 64.0%
  Adult ICU:             72.7%
  Pediatric ICU:         67.1%

Q1: Do patterns differ over time?
  → Compare blue vs orange in EACH panel month by month.
  → Lines that peak in different months or move apart = different patterns.

Q2: Is pediatric ICU proportionally more strained than adult ICU?
  → NO (pediatric ICU average is lower; ratio 0.92)


**Chart rationale:**  
I use **monthly average line charts** (national USA) because the question asks about **patterns over time** and **adult vs pediatric strain**. Lines show whether the two groups move together across months; **month labels** are easier to read than hundreds of weekly “week ending” dates.

**Why two panels?** **Inpatient** and **ICU** are different hospital resources. General bed pressure and critical-care pressure can follow different monthly patterns, so each panel answers a separate piece of the story.

**How each part of the question is answered:**
- **Different patterns?** In each panel, compare blue and orange month to month. Diverging lines = different patterns; parallel lines = similar patterns.
- **Pediatric ICU more strained proportionally?** Use the **printed averages**: if pediatric ICU mean &lt; adult ICU mean, pediatric ICU is **not** more strained overall (supported by the bottom panel’s typical gap between lines).

**Finding:** Adults run at higher occupancy than pediatric patients nationally; patterns are related but not identical across months; pediatric ICU is **not** proportionally more strained than adult ICU on average in this dataset.
